<a href="https://colab.research.google.com/github/hemanthcs34/ddpm/blob/main/ddpm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INSTALLS
# ============================================================

!pip install torch torchvision matplotlib tqdm -q

# ============================================================
# IMPORTS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from tqdm import tqdm

# ============================================================
# DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

# ============================================================
# DATASET
# ============================================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# ============================================================
# POSITIONAL ENCODING
# ============================================================

class SinusoidalPositionEmbeddings(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):

        device = time.device

        half_dim = self.dim // 2

        embeddings = torch.log(
            torch.tensor(10000.0)
        ) / (half_dim - 1)

        embeddings = torch.exp(
            torch.arange(half_dim, device=device)
            * -embeddings
        )

        embeddings = time[:, None] * embeddings[None, :]

        embeddings = torch.cat(
            (embeddings.sin(), embeddings.cos()),
            dim=-1
        )

        return embeddings

# ============================================================
# BETTER UNET
# ============================================================

class BetterUNet(nn.Module):

    def __init__(self):

        super().__init__()

        time_dim = 32

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU()
        )

        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)

        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)

        self.conv3 = nn.Conv2d(128, 64, 3, padding=1)

        self.conv4 = nn.Conv2d(64, 1, 3, padding=1)

        self.relu = nn.ReLU()

        self.time_layer = nn.Linear(time_dim, 128)

    def forward(self, x, t):

        t = self.time_mlp(t)

        t_emb = self.time_layer(t)

        t_emb = t_emb[:, :, None, None]

        x1 = self.relu(self.conv1(x))

        x2 = self.relu(self.conv2(x1))

        x2 = x2 + t_emb

        x3 = self.relu(self.conv3(x2))

        out = self.conv4(x3)

        return out

# ============================================================
# DIFFUSION
# ============================================================

class Diffusion:

    def __init__(
        self,
        noise_steps=300,
        beta_start=1e-4,
        beta_end=0.02
    ):

        self.noise_steps = noise_steps

        self.beta = torch.linspace(
            beta_start,
            beta_end,
            noise_steps
        ).to(device)

        self.alpha = 1. - self.beta

        self.alpha_hat = torch.cumprod(
            self.alpha,
            dim=0
        )

    def add_noise(self, x, t):

        sqrt_alpha_hat = torch.sqrt(
            self.alpha_hat[t]
        )[:, None, None, None]

        sqrt_one_minus_alpha_hat = torch.sqrt(
            1 - self.alpha_hat[t]
        )[:, None, None, None]

        noise = torch.randn_like(x)

        noisy_image = (
            sqrt_alpha_hat * x +
            sqrt_one_minus_alpha_hat * noise
        )

        return noisy_image, noise

# ============================================================
# MODEL
# ============================================================

model = BetterUNet().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

mse = nn.MSELoss()

diffusion = Diffusion()

# ============================================================
# TRAINING
# ============================================================

epochs = 40

print("Starting Training...")

for epoch in range(epochs):

    pbar = tqdm(loader)

    for images, _ in pbar:

        images = images.to(device)

        t = torch.randint(
            0,
            diffusion.noise_steps,
            (images.shape[0],),
            device=device
        ).float()

        noisy_images, noise = diffusion.add_noise(images, t.long())

        predicted_noise = model(noisy_images, t)

        loss = mse(noise, predicted_noise)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        pbar.set_description(
            f"Epoch {epoch+1} Loss: {loss.item():.4f}"
        )

print("Training Finished!")

# ============================================================
# SAMPLING
# ============================================================

model.eval()

with torch.no_grad():

    x = torch.randn((1,1,28,28)).to(device)

    saved_images = []

    labels = []

    for i in reversed(range(diffusion.noise_steps)):

        t = torch.tensor([i], device=device).float()

        predicted_noise = model(x, t)

        alpha = diffusion.alpha[i]

        alpha_hat = diffusion.alpha_hat[i]

        beta = diffusion.beta[i]

        if i > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (
            1 / torch.sqrt(alpha)
        ) * (
            x -
            (
                (1 - alpha) /
                torch.sqrt(1 - alpha_hat)
            ) * predicted_noise
        ) + torch.sqrt(beta) * noise

        if i % 20 == 0 or i == 0:

            saved_images.append(
                x[0].detach().cpu()
            )

            labels.append(i)

# ============================================================
# SHOW DENOISING PROCESS
# ============================================================

fig, axes = plt.subplots(
    1,
    len(saved_images),
    figsize=(20,3)
)

for idx, ax in enumerate(axes):

    ax.imshow(
        saved_images[idx].squeeze(),
        cmap="gray"
    )

    ax.set_title(f"Step {labels[idx]}")

    ax.axis("off")

plt.suptitle("DDPM Denoising Process")

plt.show()

# ============================================================
# FINAL IMAGE
# ============================================================

plt.figure(figsize=(4,4))

plt.imshow(
    saved_images[-1].squeeze(),
    cmap="gray"
)

plt.title("Final Generated Digit")

plt.axis("off")

plt.show()

Using Device: cuda


100%|██████████| 9.91M/9.91M [00:02<00:00, 4.94MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.3MB/s]


Starting Training...


Epoch 34 Loss: 0.0546:  28%|██▊       | 133/469 [00:07<00:21, 15.46it/s]